# TTS Lecture Video Generator

Generate an on-demand lecture video from **slide PDF** and **Marp narration Markdown** using Google Gemini TTS.

## Workflow
```
slides.pdf  ──┐
               ├─► [this notebook] ─► lecture_video.mp4
narration.md ──┘
```

## Prerequisites
- **Google Colab** (recommended) or local Python 3.10+ environment
- A [Google AI Studio API key](https://aistudio.google.com/) with access to `gemini-2.5-pro-preview-tts`
- Slides exported as **PDF** (from Marp or any tool)
- A **Marp `.md` file** with narration scripts embedded as HTML comments per slide:

```markdown
<!-- 
口述原稿：
Expert: Today we will cover ...
Novice: What does that mean?
-->
```

## Speaker Modes (auto-detected from narration content)
| Mode | Label | Trigger keyword | Voices |
|------|-------|----------------|--------|
| 1 | Single speaker | *(none)* | Enceladus |
| 2 | Expert × Novice | `Novice:` | Enceladus + Autonoe |
| 3 | Peer × Peer | `Peer A:` / `Peer B:` | Enceladus + Autonoe |
| 4 | Expert × Facilitator | `Facilitator:` | Enceladus + Autonoe |

In [ ]:
# Install dependencies (run once per Colab session)
!pip install -q google-genai pdf2image pydub moviepy pillow
!apt-get install -qq poppler-utils ffmpeg

## 1. Setup

In [ ]:
import os
import re
import wave
import getpass
from pathlib import Path
from typing import List, Optional

from google import genai
from google.genai import types
from pdf2image import convert_from_path
from pydub import AudioSegment
from moviepy.editor import ImageClip, AudioFileClip, concatenate_videoclips

# Mount Google Drive (optional — skip if working with local files)
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

Edit the three paths below, then run all remaining cells.

In [ ]:
# ── Input files ────────────────────────────────────────────────────────────
PDF_PATH = "/content/drive/MyDrive/path/to/slides.pdf"   # slide deck (PDF)
MD_PATH  = "/content/drive/MyDrive/path/to/narration.md" # Marp md with narrations

# ── Output ─────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/content/output"   # intermediate files + final video saved here

# ── Gemini API ─────────────────────────────────────────────────────────────
API_KEY = getpass.getpass("Google AI Studio API key: ")
client  = genai.Client(api_key=API_KEY)

MODEL_ID = "gemini-2.5-pro-preview-tts"

# ── TTS voice assignment ────────────────────────────────────────────────────
# Prebuilt Gemini voices used for each speaker role
SPEAKER_VOICES = {
    "Expert":      "Enceladus",  # single-speaker / Expert
    "Novice":      "Autonoe",
    "Peer A":      "Enceladus",
    "Peer B":      "Autonoe",
    "Facilitator": "Autonoe",
}

# Speaker configs per mode (None = single-speaker)
SPEAKER_CONFIGS = {
    1: None,
    2: {"Expert": SPEAKER_VOICES["Expert"], "Novice":      SPEAKER_VOICES["Novice"]},
    3: {"Peer A":  SPEAKER_VOICES["Peer A"], "Peer B":      SPEAKER_VOICES["Peer B"]},
    4: {"Expert": SPEAKER_VOICES["Expert"], "Facilitator": SPEAKER_VOICES["Facilitator"]},
}

# Short TTS style prompts (voice characterisation — separate from the narration-generation prompts)
VOICE_PROMPTS = {
    1: (
        "Speak as an experienced teacher. Your delivery should be clear and articulate. "
        "Use engaging intonation and vocal emphasis to capture and maintain student interest."
    ),
    2: (
        "TTS the following conversation between Expert and Novice. "
        "Make Expert sound like an experienced teacher — clear, articulate, with engaging intonation. "
        "Make Novice sound like a curious and energetic young student."
    ),
    3: (
        "TTS the following conversation between Peer A and Peer B. "
        "Both are students at the same level speaking naturally and collaboratively. "
        "Peer A is confident and articulate; Peer B is curious and engaged."
    ),
    4: (
        "TTS the following conversation between an Expert and a Facilitator. "
        "The Expert sounds authoritative and precise. "
        "The Facilitator sounds supportive and structured, guiding and summarising the discussion."
    ),
}

print("✅ Configuration ready.")

## 3. Core Functions

In [ ]:
# ── Narration extraction ────────────────────────────────────────────────────

def extract_narrations(md_path: str) -> List[str]:
    """Extract per-slide narration scripts from a Marp markdown file.

    Narrations are embedded as HTML comments of the form::

        <!-- 口述原稿:
        [script text]
        -->

    Returns a list with one entry per slide (empty string if no narration).
    """
    with open(md_path, encoding="utf-8") as f:
        content = f.read()

    # Split on Marp slide separators; slides[0] is the YAML front matter
    slides = re.split(r"\n---\n", content.strip())
    slide_bodies = slides[1:]  # drop YAML front matter

    comment_re = re.compile(r"<!--\s*(.*?)\s*-->", re.DOTALL)
    narrations: List[str] = []

    for slide in slide_bodies:
        text = ""
        for comment in reversed(comment_re.findall(slide)):  # last comment wins
            if "口述原稿" in comment:
                parts = re.split(r"口述原稿[：:]", comment, maxsplit=1)
                if len(parts) > 1:
                    raw = parts[1].strip()
                    # Remove markdown bold markers
                    raw = re.sub(r"\*\*", "", raw)
                    text = raw
                break
        narrations.append(text)

    return narrations


def detect_speaker_mode(narrations: List[str]) -> int:
    """Detect which of the four speaker modes to use.

    Searches all narration text for speaker-label keywords and returns
    the matching mode number (1–4).
    """
    combined = "\n".join(narrations)
    if "Novice:" in combined:
        return 2
    if "Peer A:" in combined or "Peer B:" in combined:
        return 3
    if "Facilitator:" in combined:
        return 4
    return 1

In [ ]:
# ── PDF → slide images ──────────────────────────────────────────────────────

def pdf_to_images(pdf_path: str, out_dir: str, dpi: int = 200) -> List[str]:
    """Rasterise each PDF page to a PNG and return a list of file paths."""
    os.makedirs(out_dir, exist_ok=True)
    pages = convert_from_path(pdf_path, dpi=dpi)
    paths = []
    for i, page in enumerate(pages):
        path = os.path.join(out_dir, f"slide_{i+1:03d}.png")
        page.save(path, "PNG")
        paths.append(path)
    print(f"   {len(paths)} pages → {out_dir}/")
    return paths

In [ ]:
# ── TTS synthesis ───────────────────────────────────────────────────────────

def _wave_file(filename: str, pcm: bytes,
               channels: int = 1, rate: int = 24000, sample_width: int = 2):
    """Write raw PCM bytes to a WAV file (24 kHz, mono, 16-bit)."""
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm)


def _build_speech_config(mode: int) -> types.SpeechConfig:
    """Build the Gemini SpeechConfig for single-speaker or multi-speaker mode."""
    speaker_config = SPEAKER_CONFIGS[mode]
    if speaker_config is None:
        return types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(
                    voice_name="Enceladus"
                )
            )
        )
    return types.SpeechConfig(
        multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
            speaker_voice_configs=[
                types.SpeakerVoiceConfig(
                    speaker=name,
                    voice_config=types.VoiceConfig(
                        prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=voice)
                    ),
                )
                for name, voice in speaker_config.items()
            ]
        )
    )


def synthesize_audio(text: str, out_path: str, mode: int, slide_idx: int) -> Optional[str]:
    """Call Gemini TTS and save the result as a padded WAV file.

    Returns the output path on success, or None if the narration is empty.
    """
    if not text.strip():
        print(f"  Slide {slide_idx+1}: no narration — skipped")
        return None

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=VOICE_PROMPTS[mode] + "\n\n" + text,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=_build_speech_config(mode),
        ),
    )

    pcm = response.candidates[0].content.parts[0].inline_data.data
    tmp = out_path + ".raw.wav"
    _wave_file(tmp, pcm)  # 24 kHz, mono, 16-bit PCM

    # Pad 1 second of silence at the beginning and end
    sound = AudioSegment.from_file(tmp)
    silence = AudioSegment.silent(duration=1000)
    (silence + sound + silence).export(out_path, format="wav")
    os.remove(tmp)

    duration = (len(sound) + 2000) / 1000
    print(f"  Slide {slide_idx+1}: {duration:.1f}s → {os.path.basename(out_path)}")
    return out_path

In [ ]:
# ── Video assembly ──────────────────────────────────────────────────────────

def assemble_video(
    image_paths: List[str],
    audio_paths: List[Optional[str]],
    out_path: str,
    fallback_duration: float = 5.0,
) -> str:
    """Combine slide images with TTS audio into a single MP4 video.

    Slides with no audio are shown for `fallback_duration` seconds.
    """
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    clips = []

    for i, (img, audio) in enumerate(zip(image_paths, audio_paths)):
        if audio and os.path.exists(audio):
            audio_clip = AudioFileClip(audio)
            duration = audio_clip.duration
            clip = ImageClip(img).set_duration(duration).set_audio(audio_clip)
        else:
            clip = ImageClip(img).set_duration(fallback_duration)
            duration = fallback_duration
        clips.append(clip)
        print(f"  Slide {i+1}: {duration:.1f}s")

    final = concatenate_videoclips(clips, method="compose")
    final.write_videofile(
        out_path, fps=30, codec="libx264", audio_codec="aac", logger=None
    )
    print(f"\n✅ Video saved: {out_path}  ({final.duration:.0f}s total)")
    return out_path

## 4. Run

Run the cell below to execute the full pipeline: PDF + MD → lecture_video.mp4

In [ ]:
img_dir   = os.path.join(OUTPUT_DIR, "images")
audio_dir = os.path.join(OUTPUT_DIR, "audio")
os.makedirs(audio_dir, exist_ok=True)

# 1. Extract narrations from the Marp MD file
print("Step 1 — Extracting narrations...")
narrations = extract_narrations(MD_PATH)
print(f"         {len(narrations)} slides in MD file")

# 2. Detect speaker mode
mode = detect_speaker_mode(narrations)
mode_name = {1: "Single speaker", 2: "Expert × Novice",
             3: "Peer × Peer", 4: "Expert × Facilitator"}[mode]
print(f"         Speaker mode: {mode} — {mode_name}")

# 3. Convert PDF to slide images
print("\nStep 2 — Converting PDF to images...")
image_paths = pdf_to_images(PDF_PATH, img_dir)

# Align slide counts (PDF pages vs. narration sections)
n = min(len(image_paths), len(narrations))
if len(image_paths) != len(narrations):
    print(f"⚠️  PDF has {len(image_paths)} pages, MD has {len(narrations)} slides → using first {n}")
image_paths = image_paths[:n]
narrations  = narrations[:n]

# 4. Generate TTS audio for each slide
print(f"\nStep 3 — Generating TTS audio ({MODEL_ID})...")
audio_paths = []
for i, text in enumerate(narrations):
    out = os.path.join(audio_dir, f"slide_{i+1:03d}.wav")
    audio_paths.append(synthesize_audio(text, out, mode, i))

# 5. Assemble video
print("\nStep 4 — Assembling video...")
video_path = os.path.join(OUTPUT_DIR, "lecture_video.mp4")
assemble_video(image_paths, audio_paths, video_path)

# 6. Download
from google.colab import files
print("\nStep 5 — Downloading...")
files.download(video_path)